# The reserved period: checks, then one scoring

[Open in Colab](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/reserved_evaluation.ipynb)

Two frozen protocols share the period 2024-01-03 to 2026-10-12: `protocols/volatility_v2.yaml` and `protocols/portfolio_v2.yaml`. **It is reserved for these specifications but previously exposed through related research**; `protocols/reserved_period_disclosure.md` lists every prior use.

This notebook runs in steps. Set `STEP` below, then Runtime → Run all. Each step includes the ones before it, and finished files are kept. Results print here; nothing is downloaded. Paste the output into the chat.

| STEP | What it does | Needs | Time |
|---|---|---|---:|
| **1** | The data checks: SPY and BIL against the recorded returns, official-open coverage, whether missing opens cluster on extreme days, quoted spreads. **Scores nothing.** For the reserved period it prints coverage counts only. | Colab Secrets | about 20 minutes |
| **2** | The baseline gate on the official open, from daily bars. **No model uses PE, and nothing after 2024-01-02 is read.** | nothing more | about 10 minutes more |
| **3** | **Scores the volatility protocol on the reserved period, once,** from daily bars. The portfolio protocol was scored on 2026-11-17. It cannot be undone. | `CONFIRM` below, typed exactly | about 15 minutes more |

**Don't save this notebook to GitHub from an old copy.** The code it runs is hash-checked by `tests/test_frozen.py`, so a change to a protocol or its code shows up.

In [1]:
STEP = 1           # 1 or 2. STEP 3 has run: both protocols are scored
CONFIRM = ''       # for STEP 3 only: 'score the reserved period'
EXPLORE = 'har_x'  # after STEP 1: 'har_x' (a better volatility model), 'signals', 'pe_long', or '' for none

In [2]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/entropy-gate-pipeline' if IN_COLAB else os.environ.get('PORTFOLIO_REPO', os.getcwd())
if IN_COLAB:
    if os.path.isdir(f'{REPO}/.git'):
        !cd {REPO} && git pull -q --ff-only
    else:
        !git clone -q https://github.com/Gaire-commits/entropy-gate-pipeline.git {REPO}
    %cd {REPO}
    !git log --oneline -1
    from google.colab import userdata
    try:
        with open('.env', 'w') as f:
            f.write(f"APCA_API_KEY_ID={userdata.get('APCA_API_KEY_ID')}\nAPCA_API_SECRET_KEY={userdata.get('APCA_API_SECRET_KEY')}\n")
        print('keys written to a local .env (never printed, never committed)')
    except Exception as exc:
        print('No Alpaca keys in Colab Secrets:', type(exc).__name__, '- add them from the key icon, then run this cell again.')
else:
    os.chdir(REPO)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
sys.path[:0] = [REPO, f'{REPO}/scripts']

from pathlib import Path
REF = os.environ.get('RESERVED_REF', '/content/reserved/reference')
WORK = os.environ.get('RESERVED_WORK', '/content/reserved')
DRIVE = '/content/drive/MyDrive/entropy-gate-pipeline-data/data'
Path(REF).mkdir(parents=True, exist_ok=True)

/content/entropy-gate-pipeline
510eb18 (HEAD -> main, origin/main, origin/HEAD) Exploration: HAR plus the signal search's features, block by block, out of sample (EXPLORE = 'har_x')
keys written to a local .env (never printed, never committed)


## Step 1: the data checks

In [3]:
!python scripts/fetch_reference.py --universe universe/sp500.csv --start 2016-01-04 --end 2026-10-13 --out {REF} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
!python scripts/official_prices.py --reference {REF}
!python scripts/protocol_checks.py --reference {REF}

wrote /content/reserved/reference/daily/daily_sip_split_nomap.parquet: 1,276,303 bars, 503 of 503 symbols
wrote /content/reserved/reference/corporate_actions.parquet: 16,794 corporate actions (42 older symbols added)
wrote /content/reserved/reference/auctions.parquet: 6,497,913 auction prints, 502 of 503 symbols
done in 753s, 1621 requests; manifest at /content/reserved/reference/manifest.json
6,497,913 auction prints for 502 symbols; 1,307,857 vendor bars with a trade, 100.00% with a listing exchange

listing exchanges (symbol-months; Nasdaq under Q): {'N': 44159, 'Q': 18534, 'Z': 67, 'V': 6, 'P': 1, nan: 1}

coverage by year (shares of vendor bars; counts only, so every year is shown):
        bars  with_listing  with_official_open  with_official_close
date                                                               
2016  115792           1.0              0.9991               0.9998
2017  116340           1.0              0.9992               0.9998
2018  117523           1.0     

## Step 2: the baseline gate on the official open

In [4]:
if STEP >= 2:
    # daily bars only (protocols/amendment_4.md): no Drive, no 5-minute cache
    !python scripts/vol_forecast.py --stage baseline --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- skipped')

STEP is 1 - skipped


## Step 3: locked

This cell scores the reserved period, once. It does nothing unless `STEP = 3` and `CONFIRM` is typed exactly, and it is not to run until Kshitiz decides. Record the outcome in `RESULTS.md` whatever it is.

In [5]:
if STEP >= 3:
    assert CONFIRM == 'score the reserved period', "set CONFIRM exactly to run this step"
    # The portfolio protocol was scored on 2026-11-17 and is not run again. Volatility on daily bars only (protocols/amendment_4.md).
    print('=' * 30, 'VOLATILITY')
    !python scripts/vol_forecast.py --stage reserved --score-reserved --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- the reserved period was not scored')

STEP is 1 - the reserved period was not scored


## Exploration (not registered tests)

Both protocols are scored, so the data from 2016 to 2026 is open for exploring. Anything found is a lead: it is written as a rule and tested on trading days after 2026-11-17. Every test run is counted. Daily bars only, no Drive.

- `'signals'`: ten simple daily features against the next day's return, the next day's open-to-close return, the next two days' return, and next-day variance beyond what volatility history predicts. About 40 tests, with a false-discovery correction.
- `'pe_long'`: PE once the long-run volatility level is in the model (run 2026-11-17; recorded in RESULTS.md).
- `'har_x'`: HAR plus the long-run level, then adding the signal search's features in blocks (the range against its usual level, recent returns, the largest return, volume shock), each measured out of sample, 2017 to 2026.

In [6]:
if EXPLORE == 'har_x':
    !python scripts/explore_har_x.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
elif EXPLORE == 'signals':
    !python scripts/explore_signals.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
elif EXPLORE in ('pe_long', True):
    !python scripts/explore_pe_long.py --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('EXPLORE is off')

rows 1,081,637, stocks 502; every model on the same rows and folds

1. Out of sample, 2017 to 2026, against the baseline (gaps: model minus baseline, negative is better; intervals 95%, blocks of 21 days):
                                       log_loss   qlike  realized_over_forecast  log_gap  log_lo  log_hi  qlike_gap  qlike_lo  qlike_hi  clark_west_t  log_gain_pct  qlike_gain_pct
HAR + long-run level (baseline)          0.6926  0.4913                  1.0544      NaN     NaN     NaN        NaN       NaN       NaN           NaN           NaN             NaN
+ range against its 63-day median        0.6931  0.4914                  1.0565   0.0005 -0.0002  0.0014     0.0001   -0.0002    0.0004       -0.5569       -0.0742         -0.0220
+ recent returns (leverage)              0.6853  0.4865                  1.0362  -0.0074 -0.0166 -0.0005    -0.0048   -0.0135    0.0009        3.7712        1.0627          0.9678
+ largest return, volume shock (full)    0.6831  0.4782                  1.